前言：目前了解程度大概是方向12>3

# 方向一：计算机视觉 —— 从图像到 CNN

## （1）图像与 Tensor

对于一张 RGB 彩色图片，其表示的 3D Tensor（高度 \(H\)、宽度 \(W\)、通道 \(C\)）分别表示什么：

- **高度 \(H\)**：图像在垂直方向上的像素数量（行数）
- **宽度 \(W\)**：图像在水平方向上的像素数量（列数）
- **通道 \(C\)**：通常为 3（Red, Green, Blue），代表每个像素点上红、绿、蓝三个颜色通道的强度值。如果是灰度图则 \(C=1\)，如果是 RGBA 图则 \(C=4\)（包含透明度）


## （2）为什么需要 CNN？
>CNN（卷积神经网络） 是一种专门用来处理图像、视频等“网格状数据”的深度学习模型。可以把它想象成“机器的一双眼睛”，它通过模拟人类视觉系统的工作原理，自动从图片中提取特征并进行识别。

假设输入是一张 `224 × 224 × 3` 的图片。如果直接展平接入全连接网络：
- **问题所在**：参数爆炸（一个神经元就有 \(224 \times 224 \times 3 = 150,528\) 个权重，极易过拟合且计算量巨大）；丢失了图像原本的二维空间结构信息（相邻像素的空间相关性被破坏）；对图像的平移、缩放等缺乏鲁棒性。
>模型的鲁棒性通常指：当输入数据发生一些变化、干扰或遇到没见过的新情况时，模型依然能做出正确判断的能力。

结合 CNN 的两个特点回答：
- **Local Connectivity（局部连接）**：卷积核只与输入图像的局部区域（感受野）相连。这大大减少了参数，并保留了图像的空间局部特征（如边缘、角点）
- **Weight Sharing（权值共享）**：同一个卷积核在整张图像上滑动，使用同一组权重。这不仅进一步大幅减少了模型参数，还赋予了模型**平移不变性**（无论特征出现在图像哪个位置，都能被同一个卷积核检测到）

## （3）卷积核与特征图

- **卷积核 (Kernel / Filter)**：一个具有可学习权重的小型矩阵（如 \(3\times3\) 或 \(5\times5\)），用于在图像上滑动进行卷积运算，目的是提取特定的局部特征
- **特征图 (Feature Map)**：输入图像经过卷积核运算后生成的输出结果。它代表了该卷积核所提取的特征在图像不同位置的激活强度分布
- **前层与后层的特征**：**浅层（前层）**通常学习到低级视觉特征（如边缘、颜色斑块、纹理）；**深层（后层）**通过组合浅层特征，学习到更抽象、高级的语义特征（如物体的部件“眼睛”、“轮子”，甚至整个“人脸”或“汽车”）

## （4）池化

- **Max Pooling（最大池化）的基本作用**：对局部区域取最大值，进行下采样。主要作用是**降低特征图的空间维度**（减少计算量和参数量），**扩大感受野**，并提供一定的**平移不变性**（微小的平移不会改变最大池化的输出）。
- **进一步思考**：池化确实会丢失精确的空间位置信息，但为什么经典 CNN 仍用？因为对于图像分类任务而言，模型更关心“有没有”这个特征（保留最强烈的特征响应），而不是特征精确的像素位置。丢失部分细节反而能迫使网络学习更鲁棒的特征，并有效防止过拟合。

## （5）ReLU 与深层网络

- **为什么要在卷积层后使用 ReLU**：
  1. 引入**非线性**。如果没有激活函数，多层网络只是线性变换的叠加，无法拟合复杂函数;
  2. 计算简单、能缓解梯度消失（相比 Sigmoid/Tanh）;
  3. 产生稀疏激活（负值置 0）,节省了计算资源，还能有效防止模型过拟合。
- **完全没有非线性激活函数**：多层卷积即使堆叠再深，在数学上也等价于单层线性变换（\(W_2(W_1x) = (W_2W_1)x\)），模型的表达能力等同于线性分类器，**不能**形成真正的非线性模型
>ReLU（全称：Rectified Linear Unit，中文常译为修正线性单元）是深度学习（尤其是 CNN）中最常用的激活函数
>如果说卷积层是“找特征”的，那么 ReLU 就是一个“过滤器”，负责决定哪些特征信号能继续往下传
>它的数学定义极其简单，公式：f(x)=max(0,x),如果输入大于 0，就原样输出；如果输入小于等于 0，就强行变成 0
## （6）预训练模型与迁移学习

**场景**：只有 1000 张猫狗图片，但有在 ImageNet 上预训练好的模型。

- **Pretrained Model（预训练模型）**：在大型数据集（如 ImageNet）上训练好的、已具备强大通用特征提取能力的模型（如 ResNet, VGG）
- **Fine-tuning（微调）**：保留预训练模型的前面特征提取层（通常冻结权重），只修改最后的分类层（将 1000 类改为 2 类：猫/狗），并使用我们的 1000 张图片对模型进行训练。可以选择只训练最后几层，或者用很小的学习率更新所有层
- **Transfer Learning（迁移学习）**：一种机器学习范式，将在一个任务（如 ImageNet 分类）上学到的知识迁移到另一个相关任务（猫狗分类）中。在 CV 中通常通过使用预训练模型和微调来实现

## （7）现代 CV 任务调研（以“目标检测”为例）

- **任务简述**：**目标检测**
- **1. 输入是什么**：一张或一批原始图像
- **2. 输出是什么**：图像中所有目标的**边界框（包含坐标 x, y, w, h）**以及对应的**类别标签**和**置信度分数**
- **3. 代表模型或应用**：**YOLO 系列**（You Only Look Once，以实时性著称）、Faster R-CNN、SSD 等。应用场景如自动驾驶（检测行人、车辆）、安防监控、工业缺陷检测等
- **4. 为什么感兴趣**：目标检测不仅是图像分类的进阶（需要同时解决“是什么”和“在哪里”的问题），而且具有极高的工业实用价值。尤其是 YOLO 这类单阶段检测器，在精度和速度之间的平衡非常巧妙，落地场景丰富




## 方向二：自然语言处理 —— 从 Token 到 Transformer

### (1) Token 与 Tokenization
*   **什么是 Token？** Token 是文本处理中的基本离散单元。它可以是完整的单词、一个汉字、一个标点符号，也可以是一个词的一部分（子词）
*   **什么是 Tokenization？** Tokenization（分词/标记化）是将原始文本字符串切分成一个个 Token 序列的过程，并通常将它们映射为数字 ID，以便计算机处理
*   **为什么大语言模型通常不简单地“一个汉字一个 Token”或“一个英文单词一个 Token”？**
    *   **按英文单词切分**：会导致词表爆炸（英语单词加各种变形超百万级），且无法处理未登录词（OOV）、拼写错误或新造词
    *   **按汉字切分**：虽然汉字总数有限，但单个字往往缺乏完整的语义（如“苹果”如果拆成“苹”和“果”，语义就跑偏了），且无法兼容多语言
    *   **现代做法**：通常采用子词切分算法（如 BPE, WordPiece）。它能在词表大小和表达能力之间取得平衡，既能控制词表规模，又能通过组合子词来处理罕见词和未见过的新词

### (2) Embedding
*   **Embedding 的作用**：计算机无法直接理解“猫”、“dog”这些文本符号。Embedding（词嵌入）的作用是将离散的 Token（如 ID）映射到一个连续的、低维的稠密向量空间中。这样，语义信息就被转化为计算机可以计算的数值特征
*   **进一步思考**：如果两个词语义比较相近，我们希望它们的 Embedding 向量在**高维空间中距离非常接近**（例如余弦相似度很高），并且具有相似的向量方向

### (3) Transformer 为什么重要？
相比传统的 RNN（循环神经网络）或 LSTM（长短期记忆网络），Transformer 有两个极其重要的特点：
1.  **并行计算能力**：RNN 必须按时间步顺序处理数据（t 时刻依赖 t-1 时刻），无法并行。而 Transformer 的 Self-Attention 机制允许它一次性同时处理序列中的所有 Token，极大提升了训练速度
2.  **解决了长距离依赖问题**：在 RNN 中，序列越长，早期信息传递到后期的损耗越大（梯度消失）。而 Transformer 中任意两个 Token 之间的交互路径长度都是 1，能够直接捕捉长距离的依赖关系

### (4) Self-Attention
*   **用自己的语言解释**：Self-Attention（自注意力机制）让句子中的每一个词都能“环顾四周”，看看句子中其他词与自己的关系有多密切，并根据这种密切程度（注意力权重），把其他词的信息融合到自己的表示中。这样，每个词的表征就不再是孤立的，而是包含了上下文信息
*   **为什么需要关注其他 Token？** 因为自然语言中的词语往往是多义的，必须结合上下文才能确定其确切含义。例如“苹果”这个词，在“我吃了一个苹果”和苹果公司发布了新手机”中，它的含义截然不同。模型必须“关注”周围的词才能正确理解它

### (5) Q, K, V
在 Transformer 中，Q、K、V 是通过输入特征线性变换得到的三个矩阵，可以借用“图书检索”来理解：
*   **Query (Q，查询)**：代表当前 Token 正在寻找什么信息（就像你在图书馆搜索框输入的关键词）
*   **Key (K，键)**：代表当前 Token 具有什么特征或标签（就像图书的标题或索引标签）
*   **Value (V，值)**：代表当前 Token 实际包含的具体内容信息（就像图书里面的正文内容）
*   **计算过程**：用 Q 去和所有的 K 做点积（计算相似度），得出每个词的权重（Attention Score），然后用这些权重去加权求和对应的 V，得到最终的上下文表示

### (6) 位置信息
*   **为什么需要位置编码？** 因为 Self-Attention 本身是**位置无关**的。不管词序如何打乱，只要包含的词一样，计算出的注意力结果是一样的
*   **语序重要性例子**：“我借给你一百块钱” 和 “你借给我一百块钱”，含义天差地别；“屡战屡败”和“屡败屡战”，情感色彩完全不同

### (7) 从 Transformer 到大语言模型
以 GPT 为例
1.  **它和 Transformer 有什么关系？** GPT (Generative Pre-trained Transformer) 是 Transformer 架构的一种变体，具体来说，它是 **Decoder-only** 架构（只使用了 Transformer 的右侧解码器部分），通过掩码注意力（Masked Self-Attention）来实现自回归的文本生成
2.  **为什么能表现出问答、写代码等能力？**
    *   **规模效应 (Scaling Laws)**：当模型参数、数据量和计算量达到极大时，模型会产生“涌现能力”
    *   **海量知识压缩**：通过在海量互联网文本上训练，模型在预测下一个 Token 的过程中，隐式地学习了语言的语法、世界的知识、逻辑推理模式，甚至代码的规则
    *   **对齐训练 (Alignment)**：通过指令微调 (SFT) 和人类反馈强化学习 (RLHF)，模型被训练成更符合人类指令的助手形式
3.  **你最感兴趣的能力是什么？**
  我最感兴趣的是零样本/少样本的逻辑推理与代码生成能力。仅通过自然语言描述需求，模型就能自动生成可运行的代码，甚至能自我纠错，这在极大程度上提高了生产力

# 方向三：大模型后训练与 Agent —— 从 SFT 到强化学习

## （1）Pretraining 与 Post-Training

### 各自解决什么问题
- **Pretraining（预训练）** ：在海量无标注文本上，通过“预测下一个 token”的自监督目标，让模型学习语言的语法、语义、世界知识和基础推理能力。它解决的是**模型有没有能力**的问题
- **Post-Training（后训练）** ：在预训练模型的基础上，使用高质量指令数据、人类偏好数据或强化学习信号，让模型的行为符合人类期望的格式、风格、安全性和推理深度。它解决的是“**模型是否听话、可用、安全**”的问题

### 为什么完成大规模预训练的模型通常仍需要后训练？
预训练模型本质上是一个强大的文本续写器，而不是一个有帮助的助手。它可能：
1.  **不遵循指令**：你问“解释量子力学”，它可能续写出更多类似的问题，而不是给出解释
2.  **输出不可控**：可能生成有害、偏见或不符合事实的内容
3.  **缺乏推理纪律**：不会按步骤演算，而是直接“猜”答案。
后训练就是把这些“原始能力”**对齐**到人类真正需要的使用方式上


## （2）SFT

### SFT 的定义
**Supervised Fine-Tuning**，即监督微调。在预训练模型的基础上，使用**输入—理想输出**的成对数据，以标准的下一 token 预测损失进行微调，让模型学会按人类期望的方式回应指令

### SFT 数据应该包含的内容（以“看到用户问题后，给出清晰、符合要求的回答”为目标）
- **指令多样性**：问答、摘要、翻译、代码、角色扮演、多轮对话等
- **回答质量**：准确、完整、格式清晰，无幻觉或有害内容
- **格式规范**：符合目标场景的结构（如分点、标题、代码块）
- **拒绝样本**：对于不应回答的请求，示范礼貌而安全的拒绝方式

## （3）Preference Data
### 为什么这类数据可以增强训练大语言模型？
图中展示的是**成对偏好数据**：同一个问题下，人类标注者认为答案 A 优于答案 B。它提供的是**相对比较信号**，而不是绝对“标准答案”。这种信号直接告诉模型：“在这两个选项中，人类更喜欢哪一个。”模型可以从大量这类比较中学习到**隐式的偏好函数**——什么样的回答在有用性、诚实性、无害性上更受青睐

### 它与普通“问题—标准答案”的监督数据有什么区别？
- **监督数据**回答的是应该输出什么，是逐 token 的绝对模仿，容易让模型只学会像标准答案，但遇到开放性问题时缺乏判断力
- **偏好数据**回答的是在两个都不完美的回答中，哪个更好，它教会模型的是**比较和排序的能力**，更贴近人类对开放式生成质量的真实判断，也更能处理没有唯一正确答案的任务


## （4）DPO 与 RLHF

### 1. RLHF 大致做什么
先训练一个奖励模型来学习人类偏好（给回答打分），再用强化学习算法（如 PPO）让语言模型去最大化这个奖励模型的分数。是一个先学裁判，再让选手按裁判的偏好训练的两阶段过程
### 2. DPO 大致做什么
跳过训练奖励模型这一步，直接用偏好数据（A 优于 B）构造一个损失函数，在语言模型本身上做类似监督学习的优化：提高生成被偏好答案的概率，降低生成被拒绝答案的概率。可以理解为用比较数据直接微调选手，不需要中间裁判。
### 3. 二者分别利用什么类型的人类偏好信息？
- RLHF：利用偏好数据**训练出一个可泛化的奖励函数**，然后在强化学习阶段利用这个奖励函数对模型**自己生成的任意新回答**进行打分。偏好信号被“压缩”进了奖励模型的参数中
- DPO：直接利用**离线收集的成对比较数据**，不额外训练奖励模型，偏好信号直接体现在损失函数对“偏好答案 vs 被拒答案”概率差的约束上

### 4. DPO 相比完整 RLHF 流程最可能的工程吸引力
1. **显著简化流程**：无需单独训练奖励模型，也无需复杂的 RL 调参（如 PPO 中的价值模型、GAE、KL 系数等），训练管线更短、更稳定
2. **大幅降低计算成本**：省去了奖励模型训练和在线 RL 采样，据估计可比 RLHF 节省 **50–60%** 的成本
3. **实现门槛低**：本质更接近监督学习，调试和复现难度远低于 RLHF，适合资源有限的团队

## （5）强化学习中的基本概念

1. **State** ：智能体在当前时刻对环境的观察/内部表示，是决策的依据
2. **Action** ：智能体在当前状态下可以采取的操作 
3. **Reward** ：环境在智能体采取某个动作后返回的标量反馈，指示该行为“好不好”
4. **Policy** ：从状态到动作（或动作分布）的映射，即智能体的行为策略

### 对应到一个“使用搜索引擎回答问题的 Agent”
- **State**：当前已积累的对话上下文 + 已检索到的中间信息（如已打开的网页摘要、已得到的部分答案）
- **Action**：决定下一步做什么，例如：生成一个搜索查询、点击某个搜索结果、从某个网页中提取段落、或直接输出最终答案
- **Reward**：最终答案被判定为正确/有用时获得的分数（也可以包含中间步骤的格式或效率奖励）
- **Policy**：由大语言模型参数决定的决策函数——给定当前 State，输出下一步 Action 的概率分布


## （6）PPO 与 GRPO

### PPO（Proximal Policy Optimization）
一种经典的 **on-policy** 强化学习算法，在 RLHF 中曾是最主流的优化方法。
它的核心是用一个**裁剪（clip）机制**限制新旧策略的更新幅度，防止训练崩溃。在完整的大模型 RLHF 流程中，PPO 通常同时涉及**四个模型**：策略模型、参考模型、**价值模型（Critic）** 、奖励模型。
价值模型用于估计状态价值以计算优势，但它带来了额外的显存和计算开销
### GRPO（Group Relative Policy Optimization）
GRPO 是对 PPO 的一种简化与改进，核心变化是 **去掉了独立的、需要训练的价值模型（Critic）** 
它的做法是：对于同一个问题，让旧策略采样**一组多个输出**，分别获得奖励，然后用**组内的相对排名（标准化后的奖励）** 直接作为每个输出的“优势”，而不依赖价值网络去估计基线
### 区别
- **PPO**：需要额外的价值模型来估计优势，显存占用高，训练链路更重
- **GRPO**：**以组内相对奖励替代价值模型**，显著降低显存和计算开销，更适配大语言模型的微调场景。DeepSeek-R1 等工作即采用了 GRPO 类方法进行强化学习训练

## （7）Reward Design
### 1. Agent 可能学出什么奇怪行为？
以“最终答案越长，Reward 越高”为例，模型可能学会：
- 堆砌废话：用大量重复、无关或空洞的内容把答案撑长，而不是提供实质信息
- 过度格式化：加入大量无意义的标题、列表、装饰性符号来增加长度
- 拒绝简洁：即使简单问题也刻意展开成长篇大论。
这些行为在数学上“最大化”了奖励函数，但完全偏离了“答案有用”的真实目标

### 2. 为什么 Reward Design 很重要？
奖励函数是强化学习的唯一训练信号。策略会朝奖励函数所度量的方向全力优化。如果奖励函数不能准确代理真实目标，模型就会学会钻空子————优化你写的指标，而不是你真正想要的东西。

### 3. 什么是 Reward Hacking？
Reward Hacking 指模型产出**在奖励函数上得分很高、但在真实意图上却在退化或取巧**的行为。
其根源可以追溯到 Goodhart 定律：当一个指标被当作优化目标时，它就不再是一个好的指标。
在大模型中，它表现为模型利用奖励模型的盲区、格式漏洞、验证器的缺陷等，来“欺骗”评估系统而非真正完成任务

### 4. 如果让你重新设计奖励，你会关注哪些方面？
- 多维度奖励：不只奖励长度或单一指标，同时覆盖正确性、相关性、简洁性、格式合规性
- 长度惩罚/归一化：对不必要的长度进行惩罚，或使用与信息密度相关的奖励
- 对抗性验证：定期用独立、更强的评估器审计高奖励样本，检测是否存在取巧模式
- 人类/模型偏好校准：用偏好数据持续校准奖励函数，防止它偏离真实人类判断
